<a href="https://colab.research.google.com/github/rabiul9137/TriageMindai/blob/main/NVIDIA_Nemotron_Model_Reasoning_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from huggingface_hub import login
login()

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-7B-Instruct"

if 'model' not in globals():
    print("Loading model and tokenizer for the first time...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        load_in_4bit=True,
        torch_dtype=torch.float16
    )
else:
    print("Model already loaded in this session. Skipping reload.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [13]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Check if the model has already been wrapped with PEFT
if 'model' in globals() and not hasattr(model, 'peft_config'):
    print("Applying LoRA configuration...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj","v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
else:
    print("LoRA already applied or model not found. Skipping PEFT setup.")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
help(prepare_model_for_kbit_training)

### What `prepare_model_for_kbit_training` does:
1. **Casts input embeddings to higher precision**: Ensures the start of the model can handle gradient calculations.
2. **Enables gradient checkpointing**: Saves memory during training by recomputing activations during the backward pass.
3. **Casts layer norms and LM head**: These layers are typically kept in `float32` to maintain stability during training.
4. **Sets up `requires_grad`**: While the 4-bit base weights remain frozen, it enables the model to 'flow' gradients back to the LoRA adapters.

**Note**: If you see a `RuntimeError` regarding `grad_fn`, it often means the model was already modified, and this setup process didn't complete correctly on a fresh model instance.

In [5]:
data = [
    {
        "text": "Question: What is 15+27?\nAnswer: Let's think step by step. 15+27=42. Final answer: \\boxed{42}"
    },
    {
        "text": "Question: If x+2=5, find x.\nAnswer: x=3. Final answer: \\boxed{3}"
    }
]

In [6]:
from datasets import Dataset

dataset = Dataset.from_list(data)

In [7]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = dataset.map(tokenize)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    fp16=True,
    report_to="none"
)

In [11]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [12]:
import os
os.environ["WANDB_DISABLED"] = "true"

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

trainer.train()

Truncating train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [15]:
print(dataset[0])
print(dataset.features)

{'text': "Question: What is 15+27?\nAnswer: Let's think step by step. 15+27=42. Final answer: \\boxed{42}", 'input_ids': [14582, 25, 3555, 374, 220, 16, 20, 10, 17, 22, 5267, 16141, 25, 6771, 594, 1744, 3019, 553, 3019, 13, 220, 16, 20, 10, 17, 22, 28, 19, 17, 13, 13023, 4226, 25, 1124, 79075, 90, 19, 17, 92, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 1

As you can see, the dataset contains `input_ids`, `attention_mask`, and `text` columns, which is suitable for `SFTTrainer`. The `input_ids` and `attention_mask` are essential for the model to process the text. The `text` column is also present from your initial data.

In [14]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Parameter: {name}, Requires Grad: {param.requires_grad}")

Parameter: base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight, Requires Grad: True
Parameter: base_model.model.base_model.m

In [16]:
model.save_pretrained("nemotron_lora")
tokenizer.save_pretrained("nemotron_lora")

('nemotron_lora/tokenizer_config.json',
 'nemotron_lora/special_tokens_map.json',
 'nemotron_lora/chat_template.jinja',
 'nemotron_lora/vocab.json',
 'nemotron_lora/merges.txt',
 'nemotron_lora/added_tokens.json',
 'nemotron_lora/tokenizer.json')

In [17]:
!zip -r submission.zip nemotron_lora

  adding: nemotron_lora/ (stored 0%)
  adding: nemotron_lora/adapter_model.safetensors (deflated 41%)
  adding: nemotron_lora/adapter_config.json (deflated 59%)
  adding: nemotron_lora/added_tokens.json (deflated 67%)
  adding: nemotron_lora/tokenizer_config.json (deflated 89%)
  adding: nemotron_lora/special_tokens_map.json (deflated 69%)
  adding: nemotron_lora/tokenizer.json (deflated 81%)
  adding: nemotron_lora/chat_template.jinja (deflated 71%)
  adding: nemotron_lora/vocab.json (deflated 61%)
  adding: nemotron_lora/README.md (deflated 65%)
  adding: nemotron_lora/merges.txt (deflated 57%)


In [18]:
from google.colab import files
files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>